In [3]:
%pip install pyprind
%pip install pandas
%pip install numpy
%pip install scikit-learn
%pip install curl-cffi
%pip install torch

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached curl_cffi-0.15.0-cp310-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (18 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (11.1 MB)
Note: you may need to restart the kernel to use updated packages.
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.28.9-py3-none-manylinux_2_18_x86_64.whl.metadata (2.0 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2

## Downloading the Datasets

In [4]:
from curl_cffi import requests

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
# Use 'impersonate' to mimic a real browser (e.g., chrome124)
response = requests.get(url, impersonate="chrome124")

# Check if the request was successful
if response.status_code == 200:
    with open("aclImdb_v1.tar.gz", "wb") as f:
        f.write(response.content)
    print("File downloaded successfully.")

File downloaded successfully.


In [5]:
import tarfile
with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
    tar.extractall()

/tmp/ipykernel_696908/219513043.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


## Preprocessing the Dataset into a more convenient format

In [6]:
import pyprind
import pandas as pd
import os
import sys

basepath = 'aclImdb'
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream=sys.stdout)

data = []

for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()
            
            data.append([txt, labels[l]]) 
            pbar.update()

df = pd.DataFrame(data, columns=['review', 'sentiment'])


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


## Store the datasets as CSV file

In [7]:
import numpy as np
np.random.seed(0)
df=df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df = df.rename(columns={"0": "review", "1": "sentiments"})
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


## Transform to tf-idf

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_tfidf_features(X_train, X_test):
    tfidf = TfidfVectorizer(use_idf=True, norm='l2', smooth_idf=True,
                            strip_accents=None, lowercase=False, preprocessor=None)
    
    X_train_tfidf = tfidf.fit_transform(X_train)  # fit only on train
    X_test_tfidf  = tfidf.transform(X_test)        # transform using same vocab
    
    return X_train_tfidf, X_test_tfidf, tfidf

## Cleaning Text Data

In [9]:
import re
def preprocessor(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = (re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    
    return text
    
df['review'] = df['review'].apply(preprocessor)

## Train/Test Split Data

In [10]:
from sklearn.model_selection import train_test_split

X = df['review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [11]:
X_train_tfidf, X_test_tfidf, tfidf = get_tfidf_features(X_train, X_test)
print(X_train_tfidf.shape)  
print(X_test_tfidf.shape)   

(35000, 89509)
(15000, 89509)


## Building an FNN Classifier

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import issparse
import torch.nn as nn
import numpy as np

class SparseDataset(Dataset):
    def __init__(self, X, y):
        self.X = X  # stays as sparse matrix
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].toarray().squeeze(), dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y

train_dataset = SparseDataset(X_train_tfidf, y_train)
test_dataset  = SparseDataset(X_test_tfidf,  y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

input_size = X_train_tfidf.shape[1]
print(f"Input size (vocab size): {input_size}")

Input size (vocab size): 89509


## Define the FNN

In [15]:
class FNN_Large(nn.Module):
    def __init__(self, input_size):
        super(FNN_Large, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

model = FNN_Large(input_size)
print(model)

FNN_Large(
  (network): Sequential(
    (0): Linear(in_features=89509, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


## Training a Baseline Model and Tuning Hyperparameters

In [16]:
import time

def train_model(model, train_loader, test_loader, lr=0.001, weight_decay=1e-4, epochs=10):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch).squeeze()
            loss  = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            train_correct, train_total = 0, 0
            for X_batch, y_batch in train_loader:
                preds = model(X_batch).squeeze()
                train_correct += ((preds >= 0.5) == y_batch).sum().item()
                train_total   += y_batch.size(0)

            test_correct, test_total = 0, 0
            for X_batch, y_batch in test_loader:
                preds = model(X_batch).squeeze()
                test_correct += ((preds >= 0.5) == y_batch).sum().item()
                test_total   += y_batch.size(0)

        train_acc = train_correct / train_total
        test_acc  = test_correct  / test_total
        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    
    elapsed = time.time() - start
    print(f"\nTime: {elapsed:.2f}s | Final Test Accuracy: {test_acc:.4f}")
    return test_acc, elapsed

In [17]:
print("=" * 50)
print("Baseline FNN: 3 hidden layers [256 → 128 → 64]")
print("=" * 50)
baseline_model = FNN_Large(input_size)
baseline_acc, baseline_time = train_model(baseline_model, train_loader, test_loader)

Baseline FNN: 3 hidden layers [256 → 128 → 64]
Epoch 1/10 | Train Acc: 0.9647 | Test Acc: 0.9027
Epoch 2/10 | Train Acc: 0.9875 | Test Acc: 0.8943
Epoch 3/10 | Train Acc: 0.9889 | Test Acc: 0.8805
Epoch 4/10 | Train Acc: 0.9861 | Test Acc: 0.8783
Epoch 5/10 | Train Acc: 0.9898 | Test Acc: 0.8790
Epoch 6/10 | Train Acc: 0.9926 | Test Acc: 0.8778
Epoch 7/10 | Train Acc: 0.9929 | Test Acc: 0.8777
Epoch 8/10 | Train Acc: 0.9938 | Test Acc: 0.8733
Epoch 9/10 | Train Acc: 0.9934 | Test Acc: 0.8743
Epoch 10/10 | Train Acc: 0.9923 | Test Acc: 0.8737

Time: 368.08s | Final Test Accuracy: 0.8737


In [18]:
results = []

configs = [
    {'lr': 0.001,  'weight_decay': 1e-4},   # default
    {'lr': 0.0005, 'weight_decay': 1e-5},   # lower lr
    {'lr': 0.01,   'weight_decay': 1e-3},   # higher lr
]

for cfg in configs:
    print(f"\n--- lr={cfg['lr']}, weight_decay={cfg['weight_decay']} ---")
    model = FNN_Large(input_size)
    acc, t = train_model(model, train_loader, test_loader,
                         lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    results.append({'lr': cfg['lr'], 'weight_decay': cfg['weight_decay'],
                    'test_acc': acc, 'time': t})


--- lr=0.001, weight_decay=0.0001 ---
Epoch 1/10 | Train Acc: 0.9618 | Test Acc: 0.8999
Epoch 2/10 | Train Acc: 0.9853 | Test Acc: 0.8906
Epoch 3/10 | Train Acc: 0.9881 | Test Acc: 0.8813
Epoch 4/10 | Train Acc: 0.9841 | Test Acc: 0.8759
Epoch 5/10 | Train Acc: 0.9809 | Test Acc: 0.8711
Epoch 6/10 | Train Acc: 0.9918 | Test Acc: 0.8777
Epoch 7/10 | Train Acc: 0.9929 | Test Acc: 0.8754
Epoch 8/10 | Train Acc: 0.9925 | Test Acc: 0.8768
Epoch 9/10 | Train Acc: 0.9918 | Test Acc: 0.8741
Epoch 10/10 | Train Acc: 0.9943 | Test Acc: 0.8735

Time: 364.94s | Final Test Accuracy: 0.8735

--- lr=0.0005, weight_decay=1e-05 ---
Epoch 1/10 | Train Acc: 0.9595 | Test Acc: 0.9063
Epoch 2/10 | Train Acc: 0.9901 | Test Acc: 0.9035
Epoch 3/10 | Train Acc: 0.9975 | Test Acc: 0.8966
Epoch 4/10 | Train Acc: 0.9993 | Test Acc: 0.8904
Epoch 5/10 | Train Acc: 0.9998 | Test Acc: 0.8860
Epoch 6/10 | Train Acc: 0.9999 | Test Acc: 0.8879
Epoch 7/10 | Train Acc: 1.0000 | Test Acc: 0.8875
Epoch 8/10 | Train Acc: 1.

In [ ]:
print("\n" + "=" * 60)
print("Comparison: FNN vs Logistic Regression (Book)")
print("=" * 60)
print(f"{'Model':<35} {'Test Acc':>10} {'Time':>10}")
print("-" * 60)
print(f"{'Logistic Regression (book)':<35} {'0.899':>10} {'5-10 min':>10}")
print(f"{'FNN Baseline [128→64]':<35} {baseline_acc:>10.3f} {baseline_time:>9.1f}s")
for r in results:
    label = f"FNN Large [256→128→64] lr={r['lr']}"
    print(f"{label:<35} {r['test_acc']:>10.3f} {r['time']:>9.1f}s")